# p641 Design Circular Deque 解析笔记

- 题号：p641
- 题目英文名：Design Circular Deque
- 题目中文名：设计循环双端队列
- 题目链接：https://leetcode.cn/problems/design-circular-deque/
- 题型：双端队列设计
- 难度：Medium
- 推荐优先级：低
- Java 原代码位置：`solutions-java/leetcode/p641_design_circular_deque/DesignCircularDeque.java`


## 1. 题目一句话总结

这道题要求我们用固定长度数组实现一个循环双端队列。

本质上考的是如何同时支持头尾两端的插入删除，并正确处理回绕。


## 2. 题目理解与约束分析

### 2.1 题目要求

实现一个固定容量的循环双端队列，支持：

- 头部插入 / 尾部插入
- 头部删除 / 尾部删除
- 查看头 / 查看尾
- 判空 / 判满

### 2.2 输入与输出

- 输入：一系列双端队列操作
- 输出：每次操作结果
- 返回结果含义：行为必须符合循环双端队列语义

### 2.3 关键约束

- 数组容量固定
- 两端都能进出
- 指针既要支持前进，也要支持后退回绕

### 2.4 示例分析

如果容量是 `3`，执行：

```text
insertLast(1)
insertLast(2)
insertFront(3)
```

那么队列里逻辑顺序会变成 `[3,1,2]`。


## 3. Java 原代码

```java
package leetcode.p641_design_circular_deque;

public class DesignCircularDeque {

    public int[] queue;
    public int head;
    public int tail;
    public int size;
    public int limit;

    public DesignCircularDeque(int n) {
        queue = new int[n];
        head = 0;
        tail = 0;
        size = 0;
        limit = n;
    }

    public boolean insertFront(int value) {
        if (isFull()) {
            return false;
        } else {
            if (isEmpty()) {
                queue[0] = value;
                head = tail = 0;
            } else {
                head = (head == 0) ? limit - 1 : head - 1;
                queue[head] = value;
            }
            size++;
            return true;
        }
    }

    public boolean insertLast(int value) {
        if (isFull()) {
            return false;
        } else {
            if (isEmpty()) {
                queue[0] = value;
                head = tail = 0;
            } else {
                tail = (tail + 1 == limit) ? 0 : tail + 1;
                queue[tail] = value;
            }
            size++;
            return true;
        }
    }

    public boolean deleteFront() {
        if (isEmpty()) {
            return false;
        } else {
            queue[head] = 0;
            head = (head + 1 == limit) ? 0 : head + 1;
            size--;
            return true;
        }
    }

    public boolean deleteLast() {
        if (isEmpty()) {
            return false;
        } else {
            queue[tail] = 0;
            tail = (tail == 0) ? limit - 1 : tail - 1;
            size--;
            return true;
        }
    }

    public int getFront() {
        return isEmpty() ? -1 : queue[head];
    }

    public int getRear() {
        return isEmpty() ? -1 : queue[tail];
    }

    public boolean isEmpty() {
        return size == 0;
    }

    public boolean isFull() {
        return size == limit;
    }
}
```


## 4. 先从 Java 原方案理解

Java 原方案和 p622 很像，但这次 `head` 和 `tail` 都表示当前有效元素的位置，而不是“下一次插入位置”。

这里最关键的两个细节是：

- 空队列第一次插入时，要把 `head = tail = 0`
- 头部插入时 `head` 向左回退，尾部插入时 `tail` 向右前进

Java 注释还专门强调了：第一次插入时必须重置头尾指针位置。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

如果不要求循环，可以用双端链表直接做。

### 5.2 为什么这里要数组循环结构

- 题目本身要求固定容量
- 需要复用数组空间
- 头尾都要支持常数时间操作

### 5.3 优化方向

Java 原方案沿用了循环数组思路，只是把头尾两端都开放出来，各自支持回绕。


## 6. 核心算法讲解

### 6.1 算法名称

- 固定数组循环双端队列
- 双向回绕指针

### 6.2 为什么想到这个算法

因为双端队列的核心就是两端进出，而固定数组要复用空间，就必须让头尾指针都能绕回数组另一端。

### 6.3 关键状态

- `queue`：底层数组
- `head`：当前队首元素位置
- `tail`：当前队尾元素位置
- `size`：当前元素数量
- `limit`：容量

### 6.4 步骤拆解

1. 插入前先判满，删除前先判空
2. 空队列第一次插入时，把头尾都放在 `0`
3. 头插时 `head` 左移回绕
4. 尾插时 `tail` 右移回绕
5. 头删时 `head` 右移回绕
6. 尾删时 `tail` 左移回绕


In [ ]:
class MyCircularDeque:
    def __init__(self, k: int):
        self.queue = [0] * k
        self.head = 0
        self.tail = 0
        self.size = 0
        self.limit = k

    def insertFront(self, value: int) -> bool:
        if self.isFull():
            return False
        if self.isEmpty():
            self.queue[0] = value
            self.head = self.tail = 0
        else:
            self.head = self.limit - 1 if self.head == 0 else self.head - 1
            self.queue[self.head] = value
        self.size += 1
        return True

    def insertLast(self, value: int) -> bool:
        if self.isFull():
            return False
        if self.isEmpty():
            self.queue[0] = value
            self.head = self.tail = 0
        else:
            self.tail = 0 if self.tail + 1 == self.limit else self.tail + 1
            self.queue[self.tail] = value
        self.size += 1
        return True

    def deleteFront(self) -> bool:
        if self.isEmpty():
            return False
        self.queue[self.head] = 0
        self.head = 0 if self.head + 1 == self.limit else self.head + 1
        self.size -= 1
        return True

    def deleteLast(self) -> bool:
        if self.isEmpty():
            return False
        self.queue[self.tail] = 0
        self.tail = self.limit - 1 if self.tail == 0 else self.tail - 1
        self.size -= 1
        return True

    def getFront(self) -> int:
        return -1 if self.isEmpty() else self.queue[self.head]

    def getRear(self) -> int:
        return -1 if self.isEmpty() else self.queue[self.tail]

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.limit


## 8. 代码逐段讲解

### 8.1 第一次插入的特殊性

Java 原解特别强调了空队列第一次插入时要重置 `head` 和 `tail`。因为这时两个指针都必须指向同一个有效元素位置。

### 8.2 头插和尾插

- `insertFront`：`head` 向左移动
- `insertLast`：`tail` 向右移动

### 8.3 头删和尾删

- `deleteFront`：`head` 向右移动
- `deleteLast`：`tail` 向左移动

因为数组是循环的，所以移动到边界后要回绕。


## 9. 复杂度分析

- 时间复杂度：所有操作都是 `O(1)`
- 空间复杂度：`O(k)`
- 为什么是这个空间复杂度：底层固定数组长度就是容量 `k`


## 10. 易错点

1. 第一次插入时没有重置 `head` 和 `tail`。
2. 头插和尾插方向弄反。
3. 删除后指针移动方向写错。
4. 空队列和满队列只靠头尾位置判断，导致状态混乱。


## 11. 面试时怎么讲

可以这样概括：

> 这题我会用固定数组配合 `head`、`tail`、`size` 来实现循环双端队列。
> 空队列第一次插入时，`head` 和 `tail` 都落在同一位置；之后头插就让 `head` 左移，尾插就让 `tail` 右移。
> 删除时则做相反方向的移动，并在边界处回绕。
> 因为没有元素搬移，所以所有操作都可以是 `O(1)`。


In [ ]:
dq = MyCircularDeque(3)
print(dq.insertLast(1))
print(dq.insertLast(2))
print(dq.insertFront(3))
print(dq.insertFront(4))
print(dq.getRear())
print(dq.isFull())
print(dq.deleteLast())
print(dq.insertFront(4))
print(dq.getFront())


## 12. Demo 输出说明

- 第四次插入应失败，说明满队列判断正确。
- 删除尾部后，再从头部插入应成功，说明两端操作和回绕都正确。
- 最终队首应是新插入的 `4`。


## 13. 一句话复盘

> 这题最关键的不是循环数组本身，而是像 Java 原解那样把头尾两端的移动方向分清楚。
